# 🚀 TAS AutoBD — Full Tutorial

**Agentic LLM-powered Automatic Business Development**  
© 2024 TAS Design Group Inc.

---

## What is AutoBD?

AutoBD is a multi-agent pipeline that automates the most time-consuming parts of B2B business development:

| Step | Agent | What it does |
|------|-------|-------------|
| 1 | **Info Agent** | Crawls the web to build a rich company profile |
| 2 | **Hypothesis Agent** | Proposes a tailored IT/AI/software solution |
| 3 | **DB Builder** | Downloads GitHub repos + web articles → FAISS vector store |
| 4 | **Proposal Agent** | RAG-powered formal HTML email proposal |

## Prerequisites

- Python 3.10+
- API keys for: **OpenAI**, **Tavily**, **SendGrid** (optional)
- A `.env` file in the project root (copy from `.env.example`)

```
OPENAI_API_KEY=sk-...
TAVILY_API_KEY=tvly-...
SENDGRID_API_KEY=SG....   # optional, only needed to send email
SENDER_EMAIL=you@company.com
```

---
## 📦 Section 1 — Installation

In [ ]:
# Install all required packages
# This cell only needs to run once per environment
import subprocess, sys

result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", "requirements.txt", "-q"],
    capture_output=True, text=True
)
if result.returncode == 0:
    print("✅ All packages installed successfully.")
else:
    print("⚠️  Some packages may have failed to install:")
    print(result.stderr[-2000:])  # show last 2000 chars of errors

---
## ⚙️ Section 2 — Configuration & Validation

In [ ]:
# Add the project root to Python path (when running notebook from a sub-directory)
import sys, os
PROJECT_ROOT = os.path.abspath(os.path.dirname("."))  # adjust if needed
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
# Load and validate configuration
from config import validate_config, DATA_DIR, GITHUB_DOWNLOAD_DIR

status = validate_config()
print("=" * 45)
print("  TAS AutoBD — Configuration Status")
print("=" * 45)
for service, ok in status.items():
    icon = "🟢" if ok else "🔴"
    label = "Configured" if ok else "MISSING — add to .env"
    print(f"  {icon}  {service.upper():<12} {label}")
print("=" * 45)
print(f"\n  Data directory : {DATA_DIR}")
print(f"  Repo cache     : {GITHUB_DOWNLOAD_DIR}")

# Block the demo if critical keys are missing
if not status["openai"] or not status["tavily"]:
    raise EnvironmentError(
        "\n\nMissing required API keys!\n"
        "Please copy .env.example → .env and fill in OPENAI_API_KEY and TAVILY_API_KEY."
    )

---
## 🔍 Section 3 — Step 1: Research Target Company

The **Info Agent** uses Tavily to run 7 parallel web searches about the target company, then processes each URL concurrently and consolidates the results into a structured profile using GPT-4o.

**What you get:**
- Company background & industry context
- Current needs and technology gaps
- Discovered email addresses for outreach
- Financial highlights

In [ ]:
# ── Configure the target company ─────────────────────────────────────────────
TARGET_COMPANY = "Fujitsu"          # Change to any company you want to pitch
ADDITIONAL_URL = ""                  # Optional: company homepage or specific page

print(f"🎯 Target company: {TARGET_COMPANY}")

In [ ]:
# ── Run the Info Agent ────────────────────────────────────────────────────────
import time
from get_info import get_company_information

print("Starting company research …\n")
t0 = time.time()

info, discovered_emails = get_company_information(TARGET_COMPANY, ADDITIONAL_URL)

elapsed = time.time() - t0
print(f"\n✅ Research complete in {elapsed:.1f}s")
print(f"📧 Discovered {len(discovered_emails)} email address(es)")

In [ ]:
# ── Display the structured company profile ────────────────────────────────────
from IPython.display import Markdown

print("=" * 60)
print(f"  Company Profile: {TARGET_COMPANY}")
print("=" * 60)
print(info.content)
print("\n" + "─" * 60)
print("Discovered emails:")
for email in discovered_emails:
    print(f"  • {email}")

In [ ]:
# Store for next steps
company_characteristics = info.content
email_list = discovered_emails

# You can also manually edit characteristics here if needed:
# company_characteristics += "\nAdditional context: ..."

---
## 💡 Section 4 — Step 2: Generate Product Hypothesis

The **Hypothesis Agent** analyses the company profile with a **human-centric** lens — considering ethical implications, user experience, sustainability, inclusivity, and privacy. It recommends ONE focused software/AI/ML solution and extracts GitHub search keywords.

In [ ]:
import asyncio
from get_hypo import get_hypothesis_idea

print("Generating product hypothesis …\n")
t0 = time.time()

idea_result, keywords = asyncio.run(get_hypothesis_idea(company_characteristics))

print(f"✅ Hypothesis generated in {time.time() - t0:.1f}s")
print(f"🔑 Search keywords: {keywords}")

In [ ]:
# Display the product idea
print("=" * 60)
print("  Product Hypothesis")
print("=" * 60)
print(idea_result.content)

In [ ]:
# Store for next steps
product_idea = idea_result.content

# You can manually edit the idea or keywords here:
# product_idea = "Custom idea text ..."
# keywords = ["custom_keyword_1", "custom_keyword_2", "custom_keyword_3"]

print(f"Keywords to use: {keywords}")

---
## 🏗️ Section 5 — Step 3: Build Knowledge Base

The **DB Builder** does three things in parallel:

1. **GitHub Search** — finds relevant open-source repositories for each keyword
2. **ZIP Download** — downloads repository archives and extracts README files
3. **Web Search** — fetches Tavily articles about the keywords

All collected text is summarised by GPT-4o and stored in a **FAISS vector store** for retrieval-augmented generation.

> ⏱️ This step typically takes **1–3 minutes** depending on API response times.

In [ ]:
import logging
# Enable INFO logging to see progress
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")

from make_db import make_db

print("Building knowledge database …\n")
t0 = time.time()

vector_db = asyncio.run(make_db(product_idea, keywords))

print(f"\n✅ Knowledge base built in {time.time() - t0:.1f}s")
print(f"📚 Vector store ready: {type(vector_db).__name__}")

In [ ]:
# Test retrieval — verify the knowledge base works
test_query = f"Key features and benefits of {product_idea[:80]}"
retriever = vector_db.as_retriever(search_kwargs={"k": 3})
retrieved_docs = retriever.invoke(test_query)

print(f"Test query: {test_query}\n")
print(f"Retrieved {len(retrieved_docs)} document(s):\n")
for i, doc in enumerate(retrieved_docs, 1):
    preview = doc.page_content[:300].replace("\n", " ")
    print(f"  [{i}] {preview}…\n")

---
## 📝 Section 6 — Step 4: Generate Business Proposal

The **Proposal Agent** runs a **3-stage pipeline**:

1. **RAG retrieval** — retrieves the most relevant evidence from the vector store
2. **Structured proposal** — generates a formal Pydantic-validated proposal with stakeholder mapping, market trends, and key features
3. **HTML email** — transforms the proposal into a polished, ready-to-send email

In [ ]:
from get_proposal import make_proposal

print("Generating proposal email …\n")
t0 = time.time()

email_html = make_proposal(product_idea, vector_db, TARGET_COMPANY)

print(f"✅ Proposal generated in {time.time() - t0:.1f}s")
print(f"📄 HTML length: {len(email_html)} chars")

In [ ]:
# Preview the HTML email in the notebook
from IPython.display import HTML
from utils import clean_html_fences

clean_email = clean_html_fences(email_html)
HTML(clean_email)

In [ ]:
# Save the proposal to disk
import os

output_path = os.path.join("data", f"proposal_{TARGET_COMPANY.replace(' ', '_')}.html")
os.makedirs("data", exist_ok=True)

with open(output_path, "w", encoding="utf-8") as f:
    f.write(clean_email)

print(f"✅ Proposal saved to: {output_path}")
print(f"   Open it in a browser for a full preview.")

---
## 📨 Section 7 — (Optional) Send Email

If you have a SendGrid API key configured, you can send the proposal directly.

> ⚠️ **The sender address must be verified in your SendGrid account.**

In [ ]:
from config import validate_config
from email_utils import send_email

cfg = validate_config()

if not cfg["sendgrid"]:
    print("ℹ️  SENDGRID_API_KEY not configured — skipping email send.")
    print("   Add it to your .env file to enable email delivery.")
else:
    # ── Set your recipient here ───────────────────────────────
    RECIPIENT_EMAIL = "recipient@example.com"   # ← change this!
    # ─────────────────────────────────────────────────────────

    if RECIPIENT_EMAIL == "recipient@example.com":
        print("⚠️  Please update RECIPIENT_EMAIL in this cell before sending.")
    else:
        success, message = send_email(
            RECIPIENT_EMAIL,
            f"Business Proposal for {TARGET_COMPANY} — TAS Design Group",
            clean_email,
        )
        if success:
            print(f"✅ {message}")
        else:
            print(f"❌ Send failed: {message}")

---
## 🏃 Section 8 — Run the Full Streamlit App

The interactive web UI wraps the entire pipeline in a polished 4-step wizard.

```bash
# From the project root
streamlit run app.py
```

The app will open at **http://localhost:8501** and provides:
- Real-time progress indicators
- Editable fields at every step
- HTML email preview with live rendering
- One-click email sending

In [ ]:
# Launch the Streamlit app from the notebook (opens in a new browser tab)
import subprocess, sys, os

proc = subprocess.Popen(
    [sys.executable, "-m", "streamlit", "run", "app.py",
     "--server.port", "8501", "--server.headless", "true"],
    cwd=os.path.dirname(os.path.abspath("app.py")),
)
print("🚀 Streamlit app started (PID", proc.pid, ")")
print("   Open: http://localhost:8501")
print("   To stop: proc.terminate() or interrupt the notebook kernel.")

---
## 🔬 Section 9 — Advanced: Customisation & Batch Processing

### 9.1 Process multiple companies

In [ ]:
# Batch pipeline — generate proposals for a list of companies
# NOTE: This makes real API calls and may incur costs.

COMPANY_LIST = [
    "Sony",
    "NTT Data",
    "Hitachi",
]

DEMO_MODE = True  # Set to False to run real API calls

if DEMO_MODE:
    print("ℹ️  DEMO_MODE=True — skipping real API calls.")
    print("   Set DEMO_MODE=False to run the full batch pipeline.")
    for company in COMPANY_LIST:
        print(f"  Would process: {company}")
else:
    results = {}
    for company in COMPANY_LIST:
        print(f"\nProcessing: {company} …")
        try:
            info, emails = get_company_information(company, "")
            idea_result, kws = asyncio.run(get_hypothesis_idea(info.content))
            db = asyncio.run(make_db(idea_result.content, kws))
            html = make_proposal(idea_result.content, db, company)
            results[company] = {"emails": emails, "proposal": clean_html_fences(html)}
            print(f"  ✅ {company} — done ({len(emails)} email(s) found)")
        except Exception as exc:
            print(f"  ❌ {company} — failed: {exc}")

    print(f"\nBatch complete. Processed {len(results)}/{len(COMPANY_LIST)} companies.")

### 9.2 Use a custom LLM model

In [ ]:
# Override the model in config without editing .env
import config

# Switch to gpt-4o-mini for cost savings
config.LLM_MODEL = "gpt-4o-mini"
config.LLM_TEMPERATURE = 0.5

# Test the new model
llm = config.get_llm()
response = llm.invoke("Say 'AutoBD ready' in one sentence.")
print(f"Model ({config.LLM_MODEL}):", response.content)

### 9.3 Inspect & query the knowledge base

In [ ]:
# Run a similarity search against the vector store we built earlier
if 'vector_db' in dir():
    queries = [
        "machine learning for manufacturing quality control",
        "data analytics dashboard for enterprise",
        "AI-powered customer service automation",
    ]

    for q in queries:
        docs = vector_db.similarity_search(q, k=2)
        print(f"Query: {q}")
        for doc in docs:
            print(f"  → {doc.page_content[:150].replace(chr(10), ' ')} …")
        print()
else:
    print("Run Section 5 first to build the vector_db.")

---
## 📊 Section 10 — Pipeline Summary

Let's print a full summary of what was generated in this session.

In [ ]:
import os

print("╔" + "═" * 58 + "╗")
print("║  TAS AutoBD — Session Summary" + " " * 28 + "║")
print("╠" + "═" * 58 + "╣")
print(f"║  Target company   : {TARGET_COMPANY:<37}║")
print(f"║  Emails found     : {len(email_list):<37}║")
print(f"║  Keywords         : {', '.join(keywords):<37}║")

# Vector DB stats
if 'vector_db' in dir():
    n_docs = vector_db.index.ntotal if hasattr(vector_db, 'index') else '?'
    print(f"║  Vectors in DB    : {str(n_docs):<37}║")

# Proposal file
proposal_path = os.path.join("data", f"proposal_{TARGET_COMPANY.replace(' ', '_')}.html")
saved = "✅ Saved" if os.path.exists(proposal_path) else "Not saved"
print(f"║  Proposal         : {saved:<37}║")
print("╚" + "═" * 58 + "╝")

print("\nNext steps:")
print("  1. Edit the proposal in data/ if needed")
print("  2. Add SENDGRID_API_KEY to .env to enable email delivery")
print("  3. Run  streamlit run app.py  for the interactive UI")
print("  4. Repeat for other target companies")